#### 7. Extend the GitHub Actions workflow to deploy to prod on merge to main using OIDC authentication(no stored secrets), including the correct permissions: id-token: write block.

```bash
name: Deploy Databricks Bundle to Prod

on:
  push:
    branches:
      - main

permissions:
  id-token: write
  contents: read

jobs:
  deploy:
    name: Deploy to Production
    runs-on: ubuntu-latest

    environment: prod

    env:
      DATABRICKS_AUTH_TYPE: github-oidc
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate production bundle
        run: databricks bundle validate --target prod

      - name: Deploy production bundle
        run: databricks bundle deploy --target prod

####8. Design a rollback plan: if a bundle deploy to prod breaks a job, what CLI commands would you run to redeploy the previous working version quickly?


```bash
# Option 1: Redeploy from a previous Git commit
git checkout <previous-working-commit-hash>
databricks bundle deploy --target prod

# Option 2: If using Git tags for releases
git checkout <previous-release-tag>
databricks bundle deploy --target prod

# Option 3: Quick rollback using Git
git revert <breaking-commit-hash>
databricks bundle deploy --target prod

# Verify the deployment
databricks bundle validate --target prod

# Check job status after rollback
databricks jobs get --job-id <job-id>
```

**Best practices for quick rollback:**
- Always tag working production deployments (e.g., `prod-v1.2.3`) for easy reference
- Keep Git history clean so previous working commits are easy to identify
- Consider using `git log --oneline` to quickly find the last working commit
- After rollback, investigate the breaking changes in a dev/staging environment before redeploying


#### 9. Write a one-page onboarding guide for a new team member explaining how a change moves from a local databricks.yml edit to running safely in production, referencing the CLI, the bundle lifecycle, and the CI/CD workflow together.

# Onboarding Guide: From Local Edit to Production Deployment

Welcome to the team! This guide walks you through how your code changes flow from your local machine to production using Databricks Asset Bundles, the Databricks CLI, and our CI/CD pipeline.

---

## The Journey of a Change

### 1. Local Development: Edit `databricks.yml`

Your journey starts with editing the `databricks.yml` file in your local project directory. This file defines your Databricks resources (jobs, pipelines, workflows) as code.

```yaml
resources:
  jobs:
    my_etl_job:
      name: "ETL Pipeline - ${bundle.target}"
      tasks:
        - task_key: process_data
          notebook_task:
            notebook_path: "./notebooks/etl.py"
```

**Key concept**: Bundle targets (dev, staging, prod) let you deploy the same code to different environments with environment-specific configurations.

---

### 2. The Bundle Lifecycle: CLI Commands

The Databricks CLI provides commands to validate, deploy, and manage your bundle:

- **`databricks bundle validate --target dev`**: Validates your `databricks.yml` syntax and configuration before deployment
- **`databricks bundle deploy --target dev`**: Deploys your resources to your dev environment (creates/updates jobs, uploads notebooks)
- **`databricks bundle run my_etl_job --target dev`**: Manually triggers a job run in dev for testing

**Best practice**: Always validate locally before deploying. Deploy to dev first, test thoroughly, then promote to production.

---

### 3. Development Workflow

1. **Create a feature branch**: `git checkout -b feature/new-transformation`
2. **Make changes**: Edit `databricks.yml`, notebooks, or Python modules
3. **Validate locally**: `databricks bundle validate --target dev`
4. **Deploy to your dev environment**: `databricks bundle deploy --target dev`
5. **Test your changes**: Run the job and verify output
6. **Commit and push**: `git add . && git commit -m "Add new transformation" && git push`
7. **Open a pull request**: Request review from the team

---

### 4. CI/CD Pipeline: Automated Safety Checks

When you open a PR, our GitHub Actions workflow automatically:

- **Validates** your bundle configuration
- **Deploys** to a staging/test environment (optional)
- **Runs** integration tests
- **Blocks merge** if validation fails

**What happens on merge to `main`**:

```yaml
permissions:
  id-token: write  # Enables secure authentication without stored secrets
  contents: read

steps:
  - Checkout code
  - Install Databricks CLI
  - Authenticate using OIDC (no secrets in repository!)
  - Validate production bundle
  - Deploy to production: databricks bundle deploy --target prod
```

**Key security feature**: OIDC authentication means GitHub gets temporary credentials directly from Databricks—no long-lived secrets to manage or rotate.

---

### 5. Production Deployment

Once your PR is approved and merged:

1. The GitHub Actions workflow triggers automatically
2. Bundle is validated against prod configuration
3. Resources are deployed to production workspace
4. Jobs are updated (existing runs continue uninterrupted)
5. You can monitor the deployment in the GitHub Actions tab

**Important**: Deployments are declarative—the bundle contains the desired state, and Databricks ensures production matches it.

---

### 6. Rollback Strategy

If something breaks in production:

```bash
# Option 1: Redeploy from the last working commit
git checkout <previous-working-commit>
databricks bundle deploy --target prod

# Option 2: Revert the breaking commit and redeploy
git revert <breaking-commit>
databricks bundle deploy --target prod
```

**Best practice**: Tag production releases (`git tag prod-v1.2.3`) so you can quickly identify stable commits.

---

## Quick Reference

| Command | Purpose |
|---------|----------|
| `databricks bundle validate --target <env>` | Check configuration syntax |
| `databricks bundle deploy --target <env>` | Deploy resources to environment |
| `databricks bundle run <job> --target <env>` | Trigger a job manually |
| `databricks jobs get --job-id <id>` | Check job status |

---

## Key Principles

✅ **Always validate before deploying**  
✅ **Test in dev before promoting to prod**  
✅ **Let CI/CD handle production deployments** (no manual deploys to prod)  
✅ **Use pull requests for code review**  
✅ **Tag working releases for easy rollback**  

---

## Getting Help

- **Documentation**: Check the `README.md` in the project root
- **Team resources**: See our internal wiki for environment URLs and access
- **Questions**: Reach out in #data-engineering Slack channel

Welcome aboard! 🚀
